# Transformer Training — Temporal Fusion Transformer on BTC
Implements a simplified Temporal Fusion Transformer for BTC price direction prediction.
Requires GPU for reasonable training time.

In [ ]:
!pip install -q torch pandas numpy scikit-learn yfinance

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

SYMBOL     = 'BTC-USD'
SEQ_LEN    = 96       # 4 days of hourly data
D_MODEL    = 64
N_HEADS    = 4
N_LAYERS   = 3
D_FF       = 256
DROPOUT    = 0.1
EPOCHS     = 40
BATCH_SIZE = 128
LR         = 5e-4
PATIENCE   = 8
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE} | d_model={D_MODEL} | heads={N_HEADS} | layers={N_LAYERS}')

In [ ]:
import yfinance as yf

df = yf.download(SYMBOL, start='2021-01-01', interval='1h', progress=False).dropna()
print(f'Downloaded {len(df)} hourly bars')

close = df['Close'].squeeze()
high  = df['High'].squeeze()
low   = df['Low'].squeeze()
vol   = df['Volume'].squeeze()

def rsi(s, n=14):
    d = s.diff(); g = d.clip(lower=0); l = -d.clip(upper=0)
    return 100 - 100/(1 + g.ewm(com=n-1,min_periods=n).mean()/(l.ewm(com=n-1,min_periods=n).mean()+1e-9))

df['ret_1h']   = close.pct_change()
df['ret_4h']   = close.pct_change(4)
df['ret_24h']  = close.pct_change(24)
df['rsi_14']   = rsi(close)
ema12          = close.ewm(span=12,adjust=False).mean()
ema26          = close.ewm(span=26,adjust=False).mean()
df['macd']     = ema12 - ema26
df['macd_sig'] = df['macd'].ewm(span=9,adjust=False).mean()
bb_mid         = close.rolling(20).mean()
bb_std         = close.rolling(20).std()
df['bb_pct']   = (close - bb_mid) / (bb_std * 2 + 1e-9)
df['bb_width'] = bb_std * 2 / (bb_mid + 1e-9)
tr             = pd.concat([high-low,(high-close.shift()).abs(),(low-close.shift()).abs()],axis=1).max(axis=1)
df['atr_14']   = tr.rolling(14).mean() / (close + 1e-9)
df['vol_ratio']= vol / (vol.rolling(24).mean() + 1e-9)
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
df['target']   = (close.shift(-1) > close).astype(int)

FEATURES = ['ret_1h','ret_4h','ret_24h','rsi_14','macd','macd_sig',
            'bb_pct','bb_width','atr_14','vol_ratio','hour_sin','hour_cos']
df = df[FEATURES + ['target']].dropna()
print(f'Samples: {len(df)} | Features: {len(FEATURES)}')

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self): return len(self.X) - self.seq_len

    def __getitem__(self, i):
        return self.X[i:i+self.seq_len], self.y[i+self.seq_len]


n = len(df)
t1, t2 = int(n * 0.70), int(n * 0.85)
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES].values)
y = df['target'].values

train_dl = DataLoader(TimeSeriesDataset(X[:t1], y[:t1], SEQ_LEN), BATCH_SIZE, shuffle=True,  drop_last=True)
val_dl   = DataLoader(TimeSeriesDataset(X[t1:t2], y[t1:t2], SEQ_LEN), BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(TimeSeriesDataset(X[t2:], y[t2:], SEQ_LEN), BATCH_SIZE, shuffle=False)
print(f'Train batches: {len(train_dl)} | Val: {len(val_dl)} | Test: {len(test_dl)}')

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TFT(nn.Module):
    """Simplified Temporal Fusion Transformer."""
    def __init__(self, n_features, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder    = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.pool       = nn.AdaptiveAvgPool1d(1)
        self.head       = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 2),
        )

    def forward(self, x):
        x = self.pos_enc(self.input_proj(x))   # (B, T, D)
        x = self.encoder(x)                    # (B, T, D)
        x = self.pool(x.permute(0, 2, 1)).squeeze(-1)  # (B, D)
        return self.head(x)


model = TFT(len(FEATURES), D_MODEL, N_HEADS, N_LAYERS, D_FF, DROPOUT).to(DEVICE)
print(f'TFT params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val = float('inf')
patience_count = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    tl = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()
    tl /= len(train_dl)

    model.eval()
    vl = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            vl += criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
    vl /= max(len(val_dl), 1)
    scheduler.step()

    if vl < best_val:
        best_val = vl
        patience_count = 0
        torch.save(model.state_dict(), '/tmp/tft_best.pt')
    else:
        patience_count += 1

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | train={tl:.4f} | val={vl:.4f} | patience={patience_count}')

    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

In [ ]:
model.load_state_dict(torch.load('/tmp/tft_best.pt', map_location=DEVICE))
model.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        preds.extend(model(xb.to(DEVICE)).argmax(-1).cpu().tolist())
        labels.extend(yb.tolist())

acc = accuracy_score(labels, preds)
print(f'Test accuracy: {acc:.4f} ({acc*100:.2f}%)')

# Save model
save_path = 'transformer_btc_1h.pt'
torch.save(model.state_dict(), save_path)
import os; print(f'Saved to {save_path} ({os.path.getsize(save_path)/1024:.1f} KB)')